# Kafka

In [ ]:
from confluent_kafka import Producer

conf = {
    "bootstrap.servers": "localhost:9092"
}
producer = Producer(conf)
producer.produce("my_topic", value="hello world")
producer.flush()


In [ ]:
from confluent_kafka import Consumer

# Config
conf = {
    'bootstrap.servers': 'localhost:9092',
    'group.id': 'test-group',
    'auto.offset.reset': 'earliest'  # start from beginning if no offset
}

consumer = Consumer(conf)
consumer.subscribe(['weather-raw'])

print("Consuming...")
while True:
    msg = consumer.poll(1.0)
    if msg is None:
        continue
    if msg.error():
        print("Error:", msg.error())
        continue
    value = msg.value().decode('utf-8')
    print(f"type: {type(value)} - {value}")

consumer.close()


# spark stream

In [23]:
%%sh
wget -O ./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar

--2025-05-03 13:20:09--  https://repo1.maven.org/maven2/org/apache/spark/spark-sql-kafka-0-10_2.13/3.2.4/spark-sql-kafka-0-10_2.13-3.2.4.jar
Resolving repo1.maven.org (repo1.maven.org)... 146.75.40.209, 151.101.196.209, 2a04:4e42:a::209, ...
Connecting to repo1.maven.org (repo1.maven.org)|146.75.40.209|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 426685 (417K) [application/java-archive]
Saving to: ‘./jars/spark-sql-kafka-0-10_2.13-3.2.4.jar’

     0K .......... .......... .......... .......... .......... 11%  193K 2s
    50K .......... .......... .......... .......... .......... 23%  177K 2s
   100K .......... .......... .......... .......... .......... 35% 13.2M 1s
   150K .......... .......... .......... .......... .......... 47% 32.9M 1s
   200K .......... .......... .......... .......... .......... 59%  196K 1s
   250K .......... .......... .......... .......... .......... 71%  613K 0s
   300K .......... .......... .......... .......... .......... 83%  

In [3]:
%%sh
spark-submit --version

Welcome to
      ____              __
     / __/__  ___ _____/ /__
    _\ \/ _ \/ _ `/ __/  '_/
   /___/ .__/\_,_/_/ /_/\_\   version 3.2.4
      /_/
                        
Using Scala version 2.12.15, OpenJDK 64-Bit Server VM, 11.0.2
Branch HEAD
Compiled by user centos on 2023-04-09T20:59:10Z
Revision 0ae10ac18298d1792828f1d59b652ef17462d76e
Url https://github.com/apache/spark
Type --help for more information.


In [4]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from contextlib import contextmanager
from pyspark.sql.types import StructType, StringType, IntegerType, TimestampType
@contextmanager
def SparkIO(app_name: str = "test_app"):    
    packages = [
        'org.apache.kafka:kafka-clients:3.2.1',
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.4"
        # "jars/spark-sql-kafka-0-10_2.13-3.2.4.jar"

    ]

    spark = SparkSession.builder\
            .master('local[*]')\
            .config('spark.app.name', f'{app_name}')\
            .config("spark.jars.packages", ",".join(packages))\
            .getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    print(f'Created SparkSession app {app_name}')
    try:
        yield spark
    except Exception as e:
        print(f'Error in SparkSession app {app_name}: {e}')
        raise
    finally:
        print(f'Stop SparkSession app {app_name}')
        spark.stop()

if __name__ == "__main__":
    with SparkIO("test_app") as spark:
        # Read from Kafka
        df = spark.readStream\
            .format("kafka")\
            .option("kafka.bootstrap.servers", "localhost:9092")\
            .option("subscribe", "weather-raw")\
            .load()
        # Print the schema
        df.printSchema()
        df.show(truncate=False)
        # Select the value column and cast it to string
        df = df.selectExpr("CAST(value AS STRING)")
        # Write to console
        query = df.writeStream\
            .outputMode("append")\
            .format("console")\
            .start()
        query.awaitTermination()

25/05/03 13:33:04 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/05/03 13:33:04 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Bootstrap broker localhost:9092 (id: -1 rack: null) disconnected
25/05/03 13:33:05 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/05/03 13:33:05 WARN NetworkClient: [Cons

Py4JError: An error occurred while calling o165.sessionState

25/05/03 13:33:07 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/05/03 13:33:07 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Bootstrap broker localhost:9092 (id: -1 rack: null) disconnected
25/05/03 13:33:08 WARN NetworkClient: [Consumer clientId=consumer-spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0-1, groupId=spark-kafka-source-b213d2c0-c41c-4935-8b93-0880983714e7-384736163-driver-0] Connection to node -1 (localhost/127.0.0.1:9092) could not be established. Broker may not be available.
25/05/03 13:33:08 WARN NetworkClient: [Cons

In [16]:
from pyspark.sql.functions import from_json, col
packages = [
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0",
    'org.apache.kafka:kafka-clients:3.2.4'
]
spark = SparkSession.builder\
         .appName("KafkaSparkStreaming")\
         .config("spark.jars.packages", ",".join(packages))\
         .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# schema = StructType()\
#     .add("current_time", StringType(), True)\
#     .add("status", StringType(), True)\
#     .add("temp_c", StringType(), True)\
#     .add("realfeel®", StringType(), True)\
#     .add("realfeel shade™", StringType(), True)\
#     .add("max uv index", StringType(), True)\
#     .add("wind", StringType(), True)\
#     .add("wind gusts", StringType(), True)\
#     .add("humidity", StringType(), True)\
#     .add("indoor humidity", StringType(), True)\
#     .add("dew point", StringType(), True)\
#     .add("pressure", StringType(), True)\
#     .add("cloud cover", StringType(), True)\
#     .add("visibility", StringType(), True)\
#     .add("cloud ceiling", StringType(), True)

df = spark.readStream\
    .format("kafka")\
    .option("kafka.bootstrap.servers", "localhost:9092")\
    .option("subscribe", "weather-raw")\
    .option("startingOffsets", "earliest")\
    .load()

# df_parsed = df.selectExpr("CAST(value AS STRING) as json_str") \
#     .select(from_json(col("json_str"), schema).alias("data")) \
#     .select("data.*")
# # df_filtered = df_parsed.filter(col("event") == "click")

# query = df_parsed.writeStream \
#     .outputMode("append") \
#     .format("console") \
#     .option("truncate", False) \
#     .start()

# query.awaitTermination()
# # {'current_time': '10:52 AM', 
# #  'status': 'Mostly sunny', 
# #  'temp_c': '33°C\n', 
# #  'realfeel®': '38°', 
# #  'realfeel shade™': '36°', 
# #  'max uv index': '5 Moderate', 
# #  'wind': 'ESE 20 km/h', 
# #  'wind gusts': '20 km/h', 
# #  'humidity': '58%', 
# #  'indoor humidity': '58% (Extremely Humid)', 
# #  'dew point': '24° C', 
# #  'pressure': '↔ 1010 mb', 'cloud cover': '30%', 'visibility': '16 km', 'cloud ceiling': '600 m'}
spark.stop()

AnalysisException:  Failed to find data source: kafka. Please deploy the application as per the deployment section of "Structured Streaming + Kafka Integration Guide".        

In [11]:
!pip show pyspark

Name: pyspark
Version: 3.2.4
Summary: Apache Spark Python API
Home-page: https://github.com/apache/spark/tree/master/python
Author: Spark Developers
Author-email: dev@spark.apache.org
License: http://www.apache.org/licenses/LICENSE-2.0
Location: /opt/conda/lib/python3.9/site-packages
Requires: py4j
Required-by: 


In [9]:
spark.stop()

In [ ]:
import requests

url = "http://localhost:8000/upload-image"

with open("ai.png", 'rb') as image_file:
    files = {'file': ('ai.png', image_file, 'image/png')}

    data = requests.post(url, files=files)


'{"image_shape":[720,1280],"total":3,"objects":[{"class_object":"person","coordinates":[100,150,200,300],"confidence":0.92,"class_id":0,"classname":"person"},{"class_object":"bicycle","coordinates":[250,400,300,500],"confidence":0.87,"class_id":1,"classname":"bicycle"},{"class_object":"bus","coordinates":[600,700,650,750],"confidence":0.95,"class_id":2,"classname":"bus"}]}'

In [2]:
from pyspark.sql import SparkSession
from pyspark import SparkConf
from contextlib import contextmanager

@contextmanager
def SparkIO(conf: SparkConf = SparkConf()):
    app_name = conf.get("spark.app.name")
    master = conf.get("spark.master")
    print(f'Create SparkSession app {app_name} with {master} mode')
    spark = SparkSession.builder.config(conf=conf).getOrCreate()
    try:
        yield spark
    except Exception:
        raise Exception
    finally:
        print(f'Stop SparkSession app {app_name}')
        spark.stop()

In [ ]:
with SparkIO()